We need to preprocess the data for when we feed it into our neural network. The steps for this are building separate audio and image folders and then zipping them up into 1 big zipfile.

In [7]:
import os
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from google.colab import drive

In [8]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
CSV_PATH = '/content/drive/MyDrive/AlbumGAN/Data/AlbumGAN_Dataset.csv' # change this to point to our finalized dataset when it's ready

os.makedirs("audio", exist_ok=True)
os.makedirs("images", exist_ok=True)

df = pd.read_csv(CSV_PATH)

In [10]:
cleaned_data = []

for index, row in df.iterrows():
    track_id = row['Track_ID']
    audio_url = row['Deezer_Audio_URL']
    image_url = row['Deezer_Image_URL']

    audio_path = f"audio/{track_id}.mp3"
    image_path = f"images/{track_id}.jpg"

    try:

        # sooo apparently the URLs from our spreadsheet expire after a few days
        # but not to worry...
        # because we have the track IDs and can re-fetch fresh URLs

        api_url = f"https://api.deezer.com/track/{track_id}"
        track_info = requests.get(api_url).json()

        fresh_audio_url = track_info.get('preview')
        fresh_image_url = track_info.get('album', {}).get('cover_xl')


        # download audio
        audio_response = requests.get(fresh_audio_url, timeout=10)
        if audio_response.status_code == 200:
            with open(audio_path, 'wb') as f:
                f.write(audio_response.content)
        else:
            raise Exception(f"Audio download failed (Status {audio_response.status_code})")


        # download and resize image
        image_response = requests.get(image_url, timeout=10)
        if image_response.status_code == 200:
            img = Image.open(BytesIO(image_response.content)).convert('RGB')
            # 128x128 is the exact standard size needed for our cDCGAN
            img = img.resize((128, 128))
            img.save(image_path)
        else:
            raise Exception(f"Image URL broken (Status {image_response.status_code})")

        cleaned_data.append({ "deezer_id": track_id, "audio_filename": f"{track_id}.mp3", "image_filename": f"{track_id}.jpg" })

    except Exception as e:
        print(f"Skipped track {track_id}: {e}")

Skipped track 3838041601: Invalid URL '': No scheme supplied. Perhaps you meant https://?


In [ ]:
clean_df = pd.DataFrame(cleaned_data)
clean_df.to_csv("cleaned_dataset.csv", index=False)

These next cells will zip and add the file to our shared drive folder.
**Do not all run this cell since we only want to transfer the zipfile once.**

In [ ]:
!zip -r -q AlbumGAN_Dataset.zip audio/ images/ cleaned_dataset.csv
!cp AlbumGAN_Dataset.zip "/content/drive/MyDrive/AlbumGAN/Data/cleaned_dataset.zip"